In [34]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

RAW  = Path("../data/raw")
PROC = Path("../data/processed"); PROC.mkdir(parents=True, exist_ok=True)
REP  = Path("../output/reports");   REP.mkdir(parents=True, exist_ok=True)

PROVINSI_TARGET = "JAWA TIMUR"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# --- config dinamis (dipakai lintas notebook) ---
CONFIG = {
    # coverage rule untuk tahun layak model
    "coverage_threshold": 0.90,        # >=90% kabupaten punya data di tahun itu
    "metric_coverage_min": 0.50,       # metrik dianggap "lengkap" jika >=50% non-null
    "min_metrik_lengkap": 3,           # minimal 3 dari 5 metrik lengkap (bukan semua)

    # threshold model
    "min_samples_regression": 10,
    "min_kab_clustering": 10,
    "label_top_share_min": 0.20,

    # outlier
    "outlier_mad_k": 5.0,              # median ± 5*MAD
    "outlier_iqr_k_fallback": 1.5,     # fallback IQR saat MAD = 0
    "suspect_scale_produksi_ton": 100_000,
    "suspect_scale_luas_ha": 100_000,  # luas wajar maksimum per kab per jenis
    "suspect_scale_rtp": 50_000,
}
pd.Series(CONFIG).to_csv(REP / "config.csv", header=["value"])

print("RAW exists:", RAW.exists())
print("Jumlah file xlsx:", len(list(RAW.glob('*.xlsx'))))
CONFIG

RAW exists: True
Jumlah file xlsx: 46


{'coverage_threshold': 0.9,
 'metric_coverage_min': 0.5,
 'min_metrik_lengkap': 3,
 'min_samples_regression': 10,
 'min_kab_clustering': 10,
 'label_top_share_min': 0.2,
 'outlier_mad_k': 5.0,
 'outlier_iqr_k_fallback': 1.5,
 'suspect_scale_produksi_ton': 100000,
 'suspect_scale_luas_ha': 100000,
 'suspect_scale_rtp': 50000}

In [35]:
def parse_number(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).strip()
    if s in {"-", "–", "—", ""}:
        return np.nan
    s = re.sub(r"\(\*\)", "", s).strip()
    s = s.replace(",", "")
    try:
        return float(s)
    except ValueError:
        return np.nan

def norm_kab(s):
    if pd.isna(s):
        return s
    s = str(s).upper().strip()
    s = re.sub(r"^(KABUPATEN|KAB\.?)\s+", "", s)
    s = re.sub(r"^(KOTA|KOTAMADYA)\s+", "KOTA ", s)
    return s.title()

KOMODITAS_SINONIM = {
    "GURAME; GURAMI": "GURAMI", "GURAME": "GURAMI",
    "MAS KOKI": "MAS", "UDANG VANAME": "UDANG",
    "LOBSTER PASIR": "LOBSTER AIR LAUT",
    "RUMPUT LAUT EUCHEUMA COTTONII": "RUMPUT LAUT",
    "IKAN LAINNYA": "LAINNYA", "BAWAL AIR TAWAR": "BAWAL",
    "LALAWAK": "TAWES",
}
def norm_komoditas(s):
    if pd.isna(s):
        return np.nan
    s = str(s).upper().strip()
    return KOMODITAS_SINONIM.get(s, s)

JENIS_BUDIDAYA_STD = {
    "JARING APUNG": "JARING APUNG",
    "JARING APUNG LAUT": "JARING APUNG",
    "JARING APUNG TAWAR": "JARING APUNG",
    "KARAMBA": "KARAMBA",
    "KOLAM": "KOLAM",
    "KOLAM AIR TENANG": "KOLAM",
    "KOLAM AIR DERAS": "KOLAM",
    "LAUT": "LAUT",
    "LAUT LAINNYA": "LAUT",
    "MINA PADI": "MINA PADI",
    "MINAPADI SAWAH": "MINA PADI",
    "MINAPADI (SAWAH)": "MINA PADI",
    "TAMBAK": "TAMBAK",
    "TAMBAK INTENSIF": "TAMBAK",
    "TAMBAK SEMI INTENSIF": "TAMBAK",
    "TAMBAK SEMI INSENTIF": "TAMBAK",
    "TAMBAK SEDERHANA": "TAMBAK",
    "RUMPUT LAUT": "RUMPUT LAUT",
    "JARING TANCAP TAWAR": "JARING TANCAP",
}
def norm_budidaya(s):
    if pd.isna(s):
        return np.nan
    s = str(s).upper().strip()
    return JENIS_BUDIDAYA_STD.get(s, s)


def flag_outlier_robust(series, k=5.0, fallback_k=1.5):
    """
    Deteksi outlier dengan median ± k*MAD.
    Jika MAD == 0 (banyak nilai identik, mis. banyak nol),
    fallback ke aturan IQR (median ± fallback_k * IQR).
    Jika IQR juga 0, tidak ada outlier.
    """
    s = pd.to_numeric(series, errors="coerce")
    med = s.median()
    mad = (s - med).abs().median()

    if pd.isna(mad) or mad == 0:
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        if pd.isna(iqr) or iqr == 0:
            return pd.Series(False, index=series.index)
        lo = q1 - fallback_k * iqr
        hi = q3 + fallback_k * iqr
        return (s < lo) | (s > hi)

    return (s - med).abs() > k * mad

In [36]:
def load_opendata_long(prefix, metrik):
    frames = []
    for f in sorted(RAW.glob(f"{prefix}*.xlsx")):
        df = pd.read_excel(f); df["source_file"] = f.name; frames.append(df)
    df = pd.concat(frames, ignore_index=True)

    df["kabupaten"]      = df["nama_kabupaten_kota"].map(norm_kab)
    df["jenis_budidaya"] = df["kategori"].map(norm_budidaya)
    df["nilai"]          = df["jumlah"].map(parse_number)
    df["tahun"]          = pd.to_numeric(df["tahun"], errors="coerce").astype("Int64")
    df["metrik"]         = metrik
    df["sumber"]         = prefix.split()[0].lower()

    df["is_missing_year"]  = False
    df["is_stale"]         = False
    df["is_aggregate_row"] = (df["kode_kabupaten_kota"].astype(str) == "0")
    df["is_suspect_scale"] = False

    if metrik == "luas_lahan_ha":
        m = (df["tahun"] == df["tahun"].max()) & (df["nilai"].fillna(0) == 0)
        df.loc[m, "is_missing_year"] = True
        df.loc[m, "nilai"] = np.nan
        # LUAS: rule bisnis saja (mayoritas nol, MAD/IQR tidak cocok)
        df["is_suspect_scale"] = df["nilai"] > CONFIG["suspect_scale_luas_ha"]


    if metrik == "rtp":
        g = df.groupby(["kabupaten", "jenis_budidaya"])["nilai"]
        df["is_stale"] = (g.transform("nunique") == 1) & (g.transform("count") >= 3)
        # RTP: rule bisnis saja (mayoritas nol, MAD/IQR tidak cocok)
        df["is_suspect_scale"] = df["nilai"] > CONFIG["suspect_scale_rtp"]

    if metrik == "produksi_ton":
        # PRODUKSI: rule bisnis saja (mayoritas nol, MAD/IQR tidak cocok)
        df["is_suspect_scale"] = df["nilai"] > CONFIG["suspect_scale_produksi_ton"]

    return df[[
        "tahun","kabupaten","jenis_budidaya","metrik","nilai","satuan",
        "sumber","source_file",
        "is_missing_year","is_stale","is_aggregate_row","is_suspect_scale",
    ]]

In [37]:
def load_bps_kegiatan():
    frames = []
    for f in sorted(RAW.glob("Produksi Perikanan Budidaya Menurut KabupatenKota dan Jenis Kegiatan*.xlsx")):
        df = pd.read_excel(f); df["source_file"] = f.name; frames.append(df)
    df = pd.concat(frames, ignore_index=True)
    df["kabupaten"] = df["Kabupaten/Kota"].map(norm_kab)
    df = df[~df["Kabupaten/Kota"].astype(str).str.upper().isin(
        {"JAWA TIMUR","KETERANGAN","ANGKA TETAP","-","NAN"})]

    col_map = {}
    for c in df.columns:
        cu = str(c).upper()
        if "PEMBESARAN" in cu:   col_map[c] = ("pembesaran", "produksi_ton")
        elif "PEMBENIHAN" in cu: col_map[c] = ("pembenihan", "volume_ribu_ekor")
        elif "IKAN HIAS" in cu:  col_map[c] = ("ikan_hias", "volume_ekor")

    long = []
    for c, (keg, met) in col_map.items():
        tmp = df[["kabupaten","source_file"]].copy()
        tmp["nilai"] = df[c].map(parse_number)
        tmp["kegiatan"] = keg; tmp["metrik"] = met; tmp["satuan"] = "mixed"
        long.append(tmp)

    out = pd.concat(long, ignore_index=True)
    out["tahun"]  = out["source_file"].str.extract(r"(\d{4})").astype(int)
    out["sumber"] = "bps_kegiatan"
    for f in ["is_missing_year","is_stale","is_aggregate_row","is_suspect_scale"]:
        out[f] = False
    return out[["tahun","kabupaten","kegiatan","metrik","nilai","satuan",
                "sumber","source_file",
                "is_missing_year","is_stale","is_aggregate_row","is_suspect_scale"]]

def load_bps_nilai_wide():
    frames = []
    for f in sorted(RAW.glob("Volume Produksi dan Nilai Produksi Perikanan Budidaya Menurut KabupatenKota*.xlsx")):
        df = pd.read_excel(f); df["source_file"] = f.name; frames.append(df)
    df = pd.concat(frames, ignore_index=True)
    df["kabupaten"] = df["Kabupaten/Kota"].map(norm_kab)
    df = df[~df["Kabupaten/Kota"].astype(str).str.upper().isin(
        {"JAWA TIMUR","KETERANGAN","-","NAN"})]

    long = []
    for c in df.columns:
        if c in {"Kabupaten/Kota","kabupaten","source_file"}:
            continue
        cu = str(c).upper()
        jb = next((v for k,v in JENIS_BUDIDAYA_STD.items() if k in cu), None)
        if jb is None or "JUMLAH" in cu: continue
        met = "nilai_rp" if "NILAI" in cu else ("produksi_kg" if ("PRODUKSI" in cu or "VOLUME" in cu) else None)
        if met is None: continue
        tmp = df[["kabupaten","source_file"]].copy()
        tmp["nilai"] = df[c].map(parse_number)
        tmp["jenis_budidaya"] = jb; tmp["metrik"] = met
        tmp["satuan"] = "kg" if met == "produksi_kg" else "Rp"
        long.append(tmp)

    out = pd.concat(long, ignore_index=True)
    out["tahun"]  = out["source_file"].str.extract(r"(\d{4})").astype(int)
    out["sumber"] = "bps_nilai"
    for f in ["is_missing_year","is_stale","is_aggregate_row","is_suspect_scale"]:
        out[f] = False
    return out[["tahun","kabupaten","jenis_budidaya","metrik","nilai","satuan",
                "sumber","source_file",
                "is_missing_year","is_stale","is_aggregate_row","is_suspect_scale"]]

In [38]:
def _load_kkp(pattern, kegiatan, vol_col, metrik_vol):
    frames = []
    for f in sorted(RAW.glob(pattern)):
        df = pd.read_excel(f); df["source_file"] = f.name; frames.append(df)
    df = pd.concat(frames, ignore_index=True)
    df.columns = [str(c).strip() for c in df.columns]
    df = df.rename(columns={"Kab / Kota":"kab_raw","Kelompok Ikan":"komoditas_raw"})
    df = df[df["Provinsi"].astype(str).str.upper() == PROVINSI_TARGET]

    df["kabupaten"] = df["kab_raw"].map(norm_kab)
    df["komoditas"] = df["komoditas_raw"].map(norm_komoditas)
    df["tahun"]     = pd.to_numeric(df["Tahun"], errors="coerce").astype("Int64")

    out = []
    for met, col in [("nilai_rp_juta","Nilai (Rp. Juta)"),(metrik_vol, vol_col)]:
        if col not in df.columns: continue
        tmp = df[["tahun","kabupaten","komoditas","source_file"]].copy()
        tmp["nilai"] = df[col].map(parse_number)
        tmp["metrik"] = met
        tmp["satuan"] = "Rp Juta" if met == "nilai_rp_juta" else vol_col
        out.append(tmp)

    res = pd.concat(out, ignore_index=True)
    res["kegiatan"]        = kegiatan
    res["sumber"]          = f"kkp_{kegiatan}"
    res["is_suspect_scale"] = False
    if kegiatan == "pembenihan":
        th_max = res["tahun"].max()
        res.loc[(res["tahun"] == th_max) & (res["metrik"] == "nilai_rp_juta"),
                "is_suspect_scale"] = True
    for f in ["is_missing_year","is_stale","is_aggregate_row"]:
        res[f] = False

    return res[["tahun","kabupaten","kegiatan","komoditas","metrik","nilai","satuan",
                "sumber","source_file",
                "is_missing_year","is_stale","is_aggregate_row","is_suspect_scale"]]

In [39]:
def load_pembudidaya():
    frames = []
    for f in sorted(RAW.glob("Pembudidaya Ikan*.xlsx")):
        df = pd.read_excel(f); df["source_file"] = f.name; frames.append(df)
    df = pd.concat(frames, ignore_index=True)
    df["kabupaten"] = df["Kabupaten / Kota"].map(norm_kab)
    df["tahun"]     = pd.to_numeric(df["Tahun"], errors="coerce").astype("Int64")
    df["metrik"]    = "pembudidaya"
    df["satuan"]    = "orang"
    df["sumber"]    = "pembudidaya"
    df["nilai"]     = df["Jumlah Pembudidaya"].map(parse_number)
    for f in ["is_missing_year","is_stale","is_aggregate_row","is_suspect_scale"]:
        df[f] = False
    return df[["tahun","kabupaten","metrik","nilai","satuan",
               "sumber","source_file",
               "is_missing_year","is_stale","is_aggregate_row","is_suspect_scale"]]

In [40]:
parts = []
parts.append(load_opendata_long("Jumlah Produksi Perikanan Budidaya Berdasarkan Jenis Budidaya","produksi_ton"))
parts.append(load_opendata_long("Jumlah RTP (Rumah Tangga Perikanan)","rtp"))
parts.append(load_opendata_long("Luas Bersih Lahan Perikanan Budidaya Berdasarkan Jenis Budidaya","luas_lahan_ha"))
parts.append(load_bps_kegiatan())
parts.append(load_bps_nilai_wide())
parts.append(_load_kkp("Produksi Perikanan Budidaya Ikan Hias*.xlsx","ikan_hias","Volume (Ekor)","volume_ekor"))
parts.append(_load_kkp("Produksi Perikanan Budidaya Pembenihan*.xlsx","pembenihan","Volume (Ribu Ekor)","volume_ribu_ekor"))
parts.append(_load_kkp("Produksi Perikanan Budidaya Pembesaran*.xlsx","pembesaran","Volume (ton)","produksi_ton"))
parts.append(load_pembudidaya())

fact = pd.concat(parts, ignore_index=True)
fact = fact[~fact["is_aggregate_row"]].drop(columns=["is_aggregate_row"])
fact.to_parquet(PROC / "fact_budidaya.parquet", index=False)
print("Fact shape:", fact.shape)
fact.head()

Fact shape: (11916, 13)


,tahun,kabupaten,jenis_budidaya,metrik,nilai,satuan,sumber,source_file,is_missing_year,is_stale,is_suspect_scale,kegiatan,komoditas
0,2020,Pacitan,JARING APUNG,produksi_ton,0.000,TON,jumlah,Jumlah Produksi Perikanan Budidaya Berdasarkan...,False,False,False,NaN,NaN
1,2020,Pacitan,KARAMBA,produksi_ton,0.000,TON,jumlah,Jumlah Produksi Perikanan Budidaya Berdasarkan...,False,False,False,NaN,NaN
2,2020,Pacitan,KOLAM,produksi_ton,455.776,TON,jumlah,Jumlah Produksi Perikanan Budidaya Berdasarkan...,False,False,False,NaN,NaN
3,2020,Pacitan,LAUT,produksi_ton,156.090,TON,jumlah,Jumlah Produksi Perikanan Budidaya Berdasarkan...,False,False,False,NaN,NaN
4,2020,Pacitan,MINA PADI,produksi_ton,2.355,TON,jumlah,Jumlah Produksi Perikanan Budidaya Berdasarkan...,False,False,False,NaN,NaN


In [41]:
print("Sumber x metrik:")
display(fact.groupby(["sumber","metrik"]).size().unstack(fill_value=0))

Sumber x metrik:


metrik,luas_lahan_ha,nilai_rp,nilai_rp_juta,pembudidaya,produksi_kg,produksi_ton,rtp,volume_ekor,volume_ribu_ekor
sumber,,,,,,,,,
bps_kegiatan,0,0,0,0,0,114,0,228,114
bps_nilai,0,912,0,0,912,0,0,0,0
jumlah,0,0,0,0,0,1368,1140,0,0
kkp_ikan_hias,0,0,725,0,0,0,0,725,0
kkp_pembenihan,0,0,607,0,0,0,0,0,607
kkp_pembesaran,0,0,1567,0,0,1567,0,0,0
luas,1140,0,0,0,0,0,0,0,0
pembudidaya,0,0,0,190,0,0,0,0,0


In [42]:
q = (fact.groupby(["sumber","tahun"])
        .agg(n=("nilai","size"),
             n_missing=("nilai", lambda s: s.isna().sum()),
             n_stale=("is_stale","sum"),
             n_missing_year=("is_missing_year","sum"),
             n_suspect=("is_suspect_scale","sum"))
        .reset_index())
q.to_csv(REP / "data_quality_report.csv", index=False)
q

,sumber,tahun,n,n_missing,n_stale,n_missing_year,n_suspect
0,bps_kegiatan,2020,152,57,0,0,0
1,bps_kegiatan,2021,152,54,0,0,0
2,bps_kegiatan,2022,152,54,0,0,0
3,bps_nilai,2023,912,684,0,0,0
4,bps_nilai,2024,912,686,0,0,0
5,jumlah,2020,456,0,124,0,1
6,jumlah,2021,456,0,124,0,74
7,jumlah,2022,456,0,124,0,1
8,jumlah,2023,456,0,124,0,1
9,jumlah,2024,456,0,124,0,0


In [43]:
print("Jumlah baris is_suspect_scale per metrik:")
display(fact.groupby("metrik")["is_suspect_scale"].sum().sort_values(ascending=False))

print("\nTop 5 luas_lahan_ha tertinggi (harus muncul yang suspect):")
display(fact[fact["metrik"]=="luas_lahan_ha"].nlargest(5, "nilai")[
    ["tahun","kabupaten","jenis_budidaya","nilai","is_suspect_scale"]])

print("\nTop 5 produksi_ton tertinggi (semua harus suspect):")
display(fact[fact["metrik"]=="produksi_ton"].nlargest(5, "nilai")[
    ["tahun","kabupaten","jenis_budidaya","nilai","is_suspect_scale"]])

print("\nJumlah nilai 0 per metrik (bantu cek MAD=0 fallback):")
display((fact["nilai"] == 0).groupby(fact["metrik"]).sum().sort_values(ascending=False))

Jumlah baris is_suspect_scale per metrik:


metrik
nilai_rp_juta       125
produksi_ton         78
luas_lahan_ha         4
pembudidaya           0
nilai_rp              0
produksi_kg           0
rtp                   0
volume_ekor           0
volume_ribu_ekor      0
Name: is_suspect_scale, dtype: int64


Top 5 luas_lahan_ha tertinggi (harus muncul yang suspect):


,tahun,kabupaten,jenis_budidaya,nilai,is_suspect_scale
3234,2023,Kota Malang,JARING APUNG,23183951.0,True
3006,2022,Kota Malang,JARING APUNG,842251.0,True
3012,2022,Lumajang,JARING APUNG,168500.0,True
3048,2022,Kota Pasuruan,JARING APUNG,120000.0,True
3222,2023,Kota Blitar,JARING APUNG,70000.0,False



Top 5 produksi_ton tertinggi (semua harus suspect):


,tahun,kabupaten,jenis_budidaya,nilai,is_suspect_scale
399,2021,Sumenep,LAUT,663714936.0,True
317,2021,Sidoarjo,TAMBAK,79108238.0,True
374,2021,Gresik,KOLAM,67610493.0,True
377,2021,Gresik,TAMBAK,63885764.0,True
370,2021,Lamongan,MINA PADI,53234312.0,True



Jumlah nilai 0 per metrik (bantu cek MAD=0 fallback):


metrik
produksi_ton        857
rtp                 683
luas_lahan_ha       591
nilai_rp_juta        95
nilai_rp              0
produksi_kg           0
pembudidaya           0
volume_ekor           0
volume_ribu_ekor      0
Name: nilai, dtype: int64

In [44]:
print("Jumlah kabupaten unik:", fact["kabupaten"].nunique())
print(sorted(fact["kabupaten"].dropna().unique()))

print("\nTop 5 produksi_ton tertinggi:")
display(fact[fact["metrik"]=="produksi_ton"].nlargest(5, "nilai")[
    ["tahun","kabupaten","jenis_budidaya","sumber","nilai","is_suspect_scale"]])

print("\nTop 5 luas_lahan_ha tertinggi:")
display(fact[fact["metrik"]=="luas_lahan_ha"].nlargest(5, "nilai")[
    ["tahun","kabupaten","jenis_budidaya","sumber","nilai","is_suspect_scale","is_missing_year"]])

print("\nBaris dengan kabupaten aneh:")
display(fact[~fact["kabupaten"].astype(str).str[0].str.isalpha()][["kabupaten"]].drop_duplicates())

Jumlah kabupaten unik: 38
['Bangkalan', 'Banyuwangi', 'Blitar', 'Bojonegoro', 'Bondowoso', 'Gresik', 'Jember', 'Jombang', 'Kediri', 'Kota Batu', 'Kota Blitar', 'Kota Kediri', 'Kota Madiun', 'Kota Malang', 'Kota Mojokerto', 'Kota Pasuruan', 'Kota Probolinggo', 'Kota Surabaya', 'Lamongan', 'Lumajang', 'Madiun', 'Magetan', 'Malang', 'Mojokerto', 'Nganjuk', 'Ngawi', 'Pacitan', 'Pamekasan', 'Pasuruan', 'Ponorogo', 'Probolinggo', 'Sampang', 'Sidoarjo', 'Situbondo', 'Sumenep', 'Trenggalek', 'Tuban', 'Tulungagung']

Top 5 produksi_ton tertinggi:


,tahun,kabupaten,jenis_budidaya,sumber,nilai,is_suspect_scale
399,2021,Sumenep,LAUT,jumlah,663714936.0,True
317,2021,Sidoarjo,TAMBAK,jumlah,79108238.0,True
374,2021,Gresik,KOLAM,jumlah,67610493.0,True
377,2021,Gresik,TAMBAK,jumlah,63885764.0,True
370,2021,Lamongan,MINA PADI,jumlah,53234312.0,True



Top 5 luas_lahan_ha tertinggi:


,tahun,kabupaten,jenis_budidaya,sumber,nilai,is_suspect_scale,is_missing_year
3234,2023,Kota Malang,JARING APUNG,luas,23183951.0,True,False
3006,2022,Kota Malang,JARING APUNG,luas,842251.0,True,False
3012,2022,Lumajang,JARING APUNG,luas,168500.0,True,False
3048,2022,Kota Pasuruan,JARING APUNG,luas,120000.0,True,False
3222,2023,Kota Blitar,JARING APUNG,luas,70000.0,False,False



Baris dengan kabupaten aneh:


,kabupaten
